# Pipeline Preprocessing Terintegrasi
## Geopolitics News × BI USD Rate

Notebook ini mengimplementasikan pipeline preprocessing untuk menggabungkan dataset **berita geopolitik** (CNBC, kolom: `keyword_matched, title, url, preview_summary, full_text, date_raw, section`) dengan **data kurs USD Bank Indonesia** (`bi-usd-rate.csv`), untuk keperluan *time-series forecasting* / *sentiment-driven financial modeling*.

**Struktur pipeline:**
1. Setup & konfigurasi
2. Load data mentah
3. Filtering data berita geopolitik
4. Cleaning data kurs BI
5. Text preprocessing (NLP pipeline)
6. Feature engineering kurs BI (Kurs Tengah, log return)
7. Standardisasi zona waktu (WIB) & aturan *market cut-off* (T+1)
8. Penyelarasan hari libur/akhir pekan (*forward-rolling alignment*)
9. Agregasi berita harian (*group-by target date*)
10. Merge dataset final & ekspor


## 1. Setup & Instalasi Dependensi

In [28]:
# Jalankan sekali saja bila package belum tersedia di environment.
# Tanda seru (!) menjalankan perintah shell dari dalam notebook.
import sys
!{sys.executable} -m pip install -q --break-system-packages pandas numpy nltk Sastrawi langdetect holidays || \
{sys.executable} -m pip install -q pandas numpy nltk Sastrawi langdetect holidays



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\ASUS\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [29]:
import re
import html
import warnings
from pathlib import Path

import holidays
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from langdetect import detect, DetectorFactory, LangDetectException

DetectorFactory.seed = 42
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Download resource NLTK yang dibutuhkan (butuh koneksi internet sekali saja)
for pkg in ["stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Gagal mengunduh '{pkg}': {e}")


## 2. Konfigurasi Pipeline

In [30]:
# Deteksi root proyek agar notebook tetap berjalan dari root proyek maupun folder src.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists() and (PROJECT_ROOT.parent / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

NEWS_PATH = PROJECT_ROOT / "data" / "raw" / "cnbc_with_published.csv"
BI_PATH   = PROJECT_ROOT / "data" / "raw" / "bi-usd-rate.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Parameter filtering berita
# ------------------------------------------------------------------
MIN_WORD_COUNT = 20                     # ambang batas minimal panjang teks
TITLE_REPEAT_WEIGHT = 2                 # bobot pengulangan judul saat penggabungan teks

RELEVANCE_KEYWORDS = [
    "geopolitics",
    "geopolitical risk",
    "geopolitical tensions",
    "geopolitical fragmentation",
    "armed conflict",
    "global conflict",
    "international conflict",
    "military tensions",
    "national security",
    "sanctions",
    "trade war",
    "tariffs",
    "embargo",
    "export controls",
    "protectionism",
    "supply chain disruption",
    "diplomacy",
    "foreign policy",
    "nato",
    "brics",
    "opec",
    "g7",
]

EXCLUDE_SECTIONS = [
    "sports", "entertainment", "television", "lifestyle", "travel",
    "food retail", "beer, wine & spirits", "college", "modern medicine",
    "life changes", "fashion", "celebrity",
]

# ------------------------------------------------------------------
# Parameter zona waktu & trading day
# ------------------------------------------------------------------
SOURCE_TZ = "America/New_York"
TARGET_TZ = "Asia/Jakarta"
MARKET_CUTOFF_HOUR = 15
START_DATE = pd.Timestamp("2021-09-01")
END_DATE = pd.Timestamp("2026-09-01 23:59:59")

print("Konfigurasi siap.")
print("NEWS_PATH:", NEWS_PATH)
print("BI_PATH:", BI_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

Konfigurasi siap.
NEWS_PATH: c:\Users\ASUS\Downloads\code\nlpproject-grestercinta\data\raw\cnbc_with_published.csv
BI_PATH: c:\Users\ASUS\Downloads\code\nlpproject-grestercinta\data\raw\bi-usd-rate.csv
OUTPUT_DIR: c:\Users\ASUS\Downloads\code\nlpproject-grestercinta\data\processed


## 3. Load Data Mentah && Selaraskan Format Data Raw

### 3.1 Load Data Mentah

In [31]:
news_raw = pd.read_csv(NEWS_PATH)
bi_raw = pd.read_csv(BI_PATH, header=None) if False else pd.read_csv(BI_PATH)

print("Berita geopolitik :", news_raw.shape)
print("Kurs USD BI       :", bi_raw.shape)
news_raw.head(5)


Berita geopolitik : (16733, 13)
Kurs USD BI       : (1202, 5)


,keyword_matched,title,url,preview_summary,full_text,date_raw,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status
0,geopolitical risk,Global shipping authorities warn of maritime trade breakdown amidgeopoliticalturmoil,https://www.cnbc.com/2026/09/08/global-shipping-iran-war-hormuz-rules.html?&qsearchterm=geopolitical risk,Global maritime authorities have warned that the emergence of “parallel systems” threatens to create a two-tier stru...,"Global maritime authorities have warned that the emergence of ""parallel systems"" threatens to create a two-tier stru...",9/8/2026 2:28:41 PM,Markets,2026-09-08T07:28:41+0000,2026-09-08,07:28 AM,0.0,2026-09-08T07:28:41+0000,ok
1,geopolitical risk,Morning Call Sheet: AI rebound meets risinggeopoliticalrisks,https://www.cnbc.com/video/2026/07/31/morning-call-sheet-ai-rebound-meets-rising-geopolitical-risks.html?&qsearchter...,"Peter Tchir, Head of Macro Strategy at Academy Securities, Thomas Martin, Senior Portfolio Manager at GLOBALT Invest...",NaN,7/31/2026 5:58:53 PM,Morning Call,NaN,NaN,NaN,NaN,NaN,not_found
2,geopolitical risk,"Global markets keep shrugging off shocks. Here’s what could break that streak, according to HSBC",https://www.cnbc.com/2026/09/08/global-markets-shrug-off-shocks-hsbc-sees-what-could-break-the-streak.html?&qsearcht...,"Global markets have shrugged off a barrage of shocks in recent years, but HSBC sees several developments that could ...","In this article Global markets have shrugged off a barrage of shocks in recent years, but HSBC sees several developm...",9/8/2026 11:02:43 AM,World Markets,2026-09-08T04:02:43+0000,2026-09-08,04:02 AM,0.0,2026-09-08T04:02:43+0000,ok
3,geopolitical risk,"Yen hovers near seven-month high, dollar steadies",https://www.cnbc.com/2026/09/08/yen-extends-rally-to-new-seven-month-high-dollar-subdued-ahead-of-cpi.html?&qsearcht...,"The Japanese yen hovered near ​a seven-month high on Tuesday, while the dollar was little changed against major peer...","In this article The Japanese yen hovered near ​a seven-month high on Tuesday, while the dollar was little changed ag...",9/8/2026 11:17:07 AM,Currencies,2026-09-08T04:17:07+0000,2026-09-08,04:17 AM,0.0,2026-09-08T04:17:07+0000,ok
4,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",6/17/2026 2:28:36 PM,Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok


In [32]:
bi_raw.head(5)


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal
0,1,1,17834.73,17657.27,9/1/2026 12:00:00 AM
1,2,1,17791.51,17614.49,8/31/2026 12:00:00 AM
2,3,1,17850.81,17673.19,8/28/2026 12:00:00 AM
3,4,1,17805.58,17628.42,8/27/2026 12:00:00 AM
4,5,1,17791.51,17614.49,8/26/2026 12:00:00 AM


### 3.2 Pisahkan Tanggal dan Jam pada dataset BI dan Geopolitik

In [33]:
# Parse timestamp mentah GEO dan pisahkan tanggal & jam dengan timezone WIB.
news_raw["date_raw_dt"] = pd.to_datetime(news_raw["date_raw"], errors="coerce")
news_raw["date_raw_dt_wib"] = (
    pd.to_datetime(news_raw["date_raw_dt"], errors="coerce", utc=True)
    .dt.tz_convert(TARGET_TZ)
    .dt.tz_localize(None)
)
news_raw["tanggal"] = news_raw["date_raw_dt_wib"].dt.strftime("%m/%d/%Y")
news_raw["jam"] = news_raw["date_raw_dt_wib"].dt.strftime("%I:%M %p")
news_raw = news_raw.drop(columns=["date_raw"], errors="ignore")

print("Kolom Berita Geopolitik:")
news_raw[["title", "tanggal", "jam", "date_raw_dt_wib"]].head(5)


Kolom Berita Geopolitik:


,title,tanggal,jam,date_raw_dt_wib
0,Global shipping authorities warn of maritime trade breakdown amidgeopoliticalturmoil,09/08/2026,09:28 PM,2026-09-08 21:28:41
1,Morning Call Sheet: AI rebound meets risinggeopoliticalrisks,08/01/2026,12:58 AM,2026-08-01 00:58:53
2,"Global markets keep shrugging off shocks. Here’s what could break that streak, according to HSBC",09/08/2026,06:02 PM,2026-09-08 18:02:43
3,"Yen hovers near seven-month high, dollar steadies",09/08/2026,06:17 PM,2026-09-08 18:17:07
4,Central banks are bringing gold reserves home asgeopoliticalrisks rise,06/17/2026,09:28 PM,2026-06-17 21:28:36


In [34]:
bi_raw["Tanggal_dt"] = pd.to_datetime(bi_raw["Tanggal"], errors="coerce")
bi_raw["tanggal"] = bi_raw["Tanggal_dt"].dt.strftime("%m/%d/%Y")
bi_raw["jam"] = bi_raw["Tanggal_dt"].dt.strftime("%I:%M %p")
bi_raw = bi_raw.drop(
    columns=["Tanggal"],
    errors="ignore"
)

print("Kolom Kurs BI:")
bi_raw.head(5)

Kolom Kurs BI:


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM
3,4,1,17805.58,17628.42,2026-08-27,08/27/2026,12:00 AM
4,5,1,17791.51,17614.49,2026-08-26,08/26/2026,12:00 AM


## 4. Filtering Data Berita Geopolitik

Tahapan:
1. **Deduplikasi** berdasarkan `url`, lalu berdasarkan kombinasi `title + tanggal + jam`.
2. **Filter teks kosong**: buang baris jika `title`, `preview_summary`, dan `full_text` semuanya kosong.
3. **Filter konten non-artikel**: buang item video, gambar, audio, podcast, galeri, dan live berdasarkan metadata URL/judul/section.
4. **Filter panjang teks**: buang berita dengan jumlah kata < `MIN_WORD_COUNT`.
5. **Filter rentang tanggal**: pertahankan berita dari 1 September 2021 sampai 1 September 2026.
6. **Filter relevansi kata kunci** dan section yang tidak relevan.

### 4.1 Deduplicate Dataset 

In [35]:
def word_count(text):
    if not isinstance(text, str):
        return 0
    return len(text.split())

news = news_raw.copy()
# Ukuran data awal
print(f"Baris awal                         : {len(news_raw)}")

# Filter rentang tanggal inklusif: 1 September 2021 - 1 September 2026
start_date = pd.Timestamp("2021-09-01")
end_date = pd.Timestamp("2026-09-01 23:59:59")
news = news[news["date_raw_dt"].between(start_date, end_date, inclusive="both")]
print(f"Setelah filter rentang 1 September 2021 - 1 September 2026 : {len(news)}")

# Setelah di drop URL
news = news.drop_duplicates(subset=["url"], keep="first")
print(f"Setelah dedup url                  : {len(news)}")

# Setelah di drop title + date_raw
news = news.drop_duplicates(subset=["title", "tanggal", "jam"], keep="first")
print(f"Setelah dedup title+date_raw       : {len(news)}")

Baris awal                         : 16733
Setelah filter rentang 1 September 2021 - 1 September 2026 : 8871
Setelah dedup url                  : 8871
Setelah dedup title+date_raw       : 7084


### 4.2 Filter Teks Kosong dan Konten Non-Artikel

In [36]:
# Buang baris yang tidak memiliki isi teks pada ketiga sumber konten.
text_columns = ["title", "preview_summary", "full_text"]
text_content = news[text_columns].fillna("").astype(str).agg(" ".join, axis=1).str.strip()
mask_empty = text_content.eq("")
news = news.loc[~mask_empty].copy()
print(f"Setelah drop teks kosong             : {len(news)}")

# Buang item multimedia berdasarkan URL, judul, dan section.
non_article_pattern = r"\b(video|videos|image|images|photo|photos|foto|gambar|audio|podcast|gallery|galeri|live)\b|\.(mp4|mov|avi|mp3|wav|jpg|jpeg|png)(?:$|[?#])"
content_metadata = news[["url", "title", "section"]].fillna("").astype(str).agg(" ".join, axis=1)
mask_non_article = content_metadata.str.contains(non_article_pattern, case=False, regex=True, na=False)
news = news.loc[~mask_non_article].copy()
print(f"Setelah drop video/gambar/audio      : {len(news)}")

news.head(5)

Setelah drop teks kosong             : 7084
Setelah drop video/gambar/audio      : 6214


,keyword_matched,title,url,preview_summary,full_text,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status,date_raw_dt,date_raw_dt_wib,tanggal,jam
4,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok,2026-06-17 14:28:36,2026-06-17 21:28:36,06/17/2026,09:28 PM
8,geopolitical risk,Oil rises 2% as White House says no US-Iran talks happening,https://www.cnbc.com/2026/08/27/oil-prices-extend-losses-on-expectations-talks-to-ease-middle-east-supply-woes.html?...,"Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomatic efforts by ot...","In this article Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomat...",Oil,2026-08-27T02:49:03+0000,2026-08-27,02:49 AM,0.0,2026-08-27T02:49:03+0000,ok,2026-08-27 09:49:03,2026-08-27 16:49:03,08/27/2026,04:49 PM
9,geopolitical risk,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,https://www.cnbc.com/2026/07/21/jpmorgan-chase-ceo-jamie-dimon-market-risk.html?&qsearchterm=geopolitical risk,JPMorgan Chase CEO Jamie Dimon said investors are underestimating the risks facing the global economy and that he wo...,In this article JPMorgan ChaseCEOJamie Dimonsaid investors are underestimating the risks facing the global economy a...,Finance,2026-07-20T23:01:01+0000,2026-07-20,11:01 PM,0.0,2026-07-20T23:01:01+0000,ok,2026-07-21 06:01:01,2026-07-21 13:01:01,07/21/2026,01:01 PM
13,geopolitical risk,"Citi Wealth warns markets may be ‘uncomfortably strong’ amid mountinggeopolitical, inflation risks",https://www.cnbc.com/2026/05/18/citi-wealth-warns-markets-may-be-uncomfortably-strong-as-risks-mount.html?&qsearchte...,"Global markets may be due for a period of consolidation after a sharp rally in equities, even as the longer-term out...",NaN,Pro: Analysis,2026-05-18T05:37:18+0000,2026-05-18,05:37 AM,0.0,2026-05-18T05:37:18+0000,ok,2026-05-18 12:37:18,2026-05-18 19:37:18,05/18/2026,07:37 PM
15,geopolitical risk,Where fixed income investors are finding yield asgeopoliticalriskrattles markets,https://www.cnbc.com/2026/04/10/where-fixed-income-investors-are-finding-yield-as-geopolitical-risk-rattles-markets....,"As the Iran war shakes up markets, strategists say there are still plenty of sources of relatively safe yield for in...",NaN,Pro: Income Investing,2026-04-10T19:25:17+0000,2026-04-10,07:25 PM,0.0,2026-04-10T19:25:17+0000,ok,2026-04-11 02:25:17,2026-04-11 09:25:17,04/11/2026,09:25 AM


### 4.3 Filter Panjang Teks Berita < 20 Kata

In [37]:
# Filter panjang teks minimal (gunakan full_text, fallback ke preview_summary)
effective_text = news["full_text"].fillna(news["preview_summary"])
news = news.loc[effective_text.apply(word_count) >= MIN_WORD_COUNT]
print(f"Setelah filter panjang teks min     : {len(news)}")

Setelah filter panjang teks min     : 6212


### 4.4 Filter Relefansi & Section Menggunakan Keyword

In [38]:
def is_relevant(row):
    """Berita dianggap relevan jika keyword_matched terisi ATAU teks memuat
    salah satu kata kunci geopolitik/ekonomi pada RELEVANCE_KEYWORDS."""
    if isinstance(row["keyword_matched"], str) and row["keyword_matched"].strip():
        return True
    haystack = f"{row.get('title', '')} {row.get('preview_summary', '')}".lower()
    return any(kw in haystack for kw in RELEVANCE_KEYWORDS)

mask_relevant = news.apply(is_relevant, axis=1)
news = news.loc[mask_relevant]

print(f"Setelah filter relevansi kata kunci : {len(news)}")

# 4.5 Filter kategori/section yang tidak relevan
section_lower = news["section"].fillna("").str.lower()
mask_section = ~section_lower.isin(EXCLUDE_SECTIONS)
news = news.loc[mask_section].reset_index(drop=True)

print(f"Setelah filter kategori/section     : {len(news)}")

news.head(3)


Setelah filter relevansi kata kunci : 6212
Setelah filter kategori/section     : 6196


,keyword_matched,title,url,preview_summary,full_text,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status,date_raw_dt,date_raw_dt_wib,tanggal,jam
0,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok,2026-06-17 14:28:36,2026-06-17 21:28:36,06/17/2026,09:28 PM
1,geopolitical risk,Oil rises 2% as White House says no US-Iran talks happening,https://www.cnbc.com/2026/08/27/oil-prices-extend-losses-on-expectations-talks-to-ease-middle-east-supply-woes.html?...,"Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomatic efforts by ot...","In this article Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomat...",Oil,2026-08-27T02:49:03+0000,2026-08-27,02:49 AM,0.0,2026-08-27T02:49:03+0000,ok,2026-08-27 09:49:03,2026-08-27 16:49:03,08/27/2026,04:49 PM
2,geopolitical risk,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,https://www.cnbc.com/2026/07/21/jpmorgan-chase-ceo-jamie-dimon-market-risk.html?&qsearchterm=geopolitical risk,JPMorgan Chase CEO Jamie Dimon said investors are underestimating the risks facing the global economy and that he wo...,In this article JPMorgan ChaseCEOJamie Dimonsaid investors are underestimating the risks facing the global economy a...,Finance,2026-07-20T23:01:01+0000,2026-07-20,11:01 PM,0.0,2026-07-20T23:01:01+0000,ok,2026-07-21 06:01:01,2026-07-21 13:01:01,07/21/2026,01:01 PM


In [39]:
jumlah_nan = news["full_text"].isna().sum()

print("Jumlah full_text NaN:", jumlah_nan)

news_nan = news.loc[news["full_text"].isna()]

news_nan[
    ["url", "title", "preview_summary", "full_text"]
].head()

news = news.dropna(subset=["full_text"]).reset_index(drop=True)

print("Jumlah baris setelah drop NaN:", len(news))
print("Sisa NaN pada full_text:", news["full_text"].isna().sum())

Jumlah full_text NaN: 1319
Jumlah baris setelah drop NaN: 4877
Sisa NaN pada full_text: 0


## 5. Cleaning Data Kurs USD BI

Tahapan:
1. **Bersihkan metadata header/footer** — ambil hanya baris tabel riil dengan kolom `NO, Nilai, Kurs Jual, Kurs Beli, Tanggal` (fungsi dibuat generik agar tetap berfungsi meski file sumber memuat baris metadata di awal/akhir).
2. **Bersihkan karakter numerik** (pemisah ribuan) & konversi ke `float`.
3. **Filter outlier/anomali**: buang baris jika `Kurs Jual ≤ Kurs Beli`, atau bernilai nol/negatif.
4. Parse kolom `Tanggal` menjadi `datetime`.


In [40]:
EXPECTED_COLS = ["NO", "Nilai", "Kurs Jual", "Kurs Beli", "tanggal", "jam"]


def clean_bi_header_footer(df_raw: pd.DataFrame) -> pd.DataFrame:
    """Ambil baris data BI dan abaikan metadata atau footer."""
    df = df_raw.copy()
    available_cols = [column for column in EXPECTED_COLS if column in df.columns]

    if len(available_cols) < len(EXPECTED_COLS):
        header_row_idx = None
        for index, row in df_raw.iterrows():
            row_values = {str(value).strip() for value in row.values}
            if set(EXPECTED_COLS).issubset(row_values):
                header_row_idx = index
                break

        if header_row_idx is not None:
            df = df_raw.iloc[header_row_idx + 1:].copy()
            df.columns = [str(value).strip() for value in df_raw.iloc[header_row_idx]]

    return df.dropna(how="all").reset_index(drop=True)


bi = clean_bi_header_footer(bi_raw)
bi = bi[[column for column in EXPECTED_COLS if column in bi.columns]].copy()

# Gabungkan tanggal dan jam untuk satu kolom datetime kanonik.
bi["Tanggal"] = pd.to_datetime(
    bi["tanggal"].astype(str).str.strip() + " " + bi["jam"].astype(str).str.strip(),
    format="%m/%d/%Y %I:%M %p",
    errors="coerce",
)

print(bi.shape)
bi.head(3)

(1202, 7)


,NO,Nilai,Kurs Jual,Kurs Beli,tanggal,jam,Tanggal
0,1,1,17834.73,17657.27,09/01/2026,12:00 AM,2026-09-01
1,2,1,17791.51,17614.49,08/31/2026,12:00 AM,2026-08-31
2,3,1,17850.81,17673.19,08/28/2026,12:00 AM,2026-08-28


In [41]:
def clean_numeric(series: pd.Series) -> pd.Series:
    """Hapus pemisah ribuan dan konversi nilai ke float."""
    cleaned = series.astype(str).str.replace(",", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")


bi["Kurs Jual"] = clean_numeric(bi["Kurs Jual"])
bi["Kurs Beli"] = clean_numeric(bi["Kurs Beli"])
bi["NO"] = pd.to_numeric(bi["NO"], errors="coerce")
bi["Nilai"] = pd.to_numeric(bi["Nilai"], errors="coerce")

print(f"Baris awal BI                 : {len(bi)}")

# Pertahankan hanya rentang 1 September 2021 sampai 1 September 2026.
bi = bi.loc[bi["Tanggal"].between(start_date, end_date, inclusive="both")]
print(f"Setelah filter rentang tanggal: {len(bi)}")

mask_valid_numeric = (
    bi["Kurs Jual"].notna()
    & bi["Kurs Beli"].notna()
    & (bi["Kurs Jual"] > 0)
    & (bi["Kurs Beli"] > 0)
)
bi = bi.loc[mask_valid_numeric]
print(f"Setelah filter numerik valid  : {len(bi)}")

mask_valid_spread = bi["Kurs Jual"] > bi["Kurs Beli"]
bi = bi.loc[mask_valid_spread]
print(f"Setelah filter Jual > Beli    : {len(bi)}")

bi = (
    bi.sort_values("Tanggal")
    .drop_duplicates(subset=["Tanggal"], keep="last")
    .reset_index(drop=True)
)
print(f"Setelah dedup tanggal         : {len(bi)}")

bi.head(3)

Baris awal BI                 : 1202
Setelah filter rentang tanggal: 1202
Setelah filter numerik valid  : 1202
Setelah filter Jual > Beli    : 1202
Setelah dedup tanggal         : 1202


,NO,Nilai,Kurs Jual,Kurs Beli,tanggal,jam,Tanggal
0,1202,1,14377.53,14234.47,09/01/2021,12:00 AM,2021-09-01
1,1201,1,14355.42,14212.58,09/02/2021,12:00 AM,2021-09-02
2,1200,1,14352.41,14209.60,09/03/2021,12:00 AM,2021-09-03


## 6. Text Preprocessing (NLP Pipeline)


### 6.1 Penggabungan Teks (Title Weighting)

`title` (diulang `TITLE_REPEAT_WEIGHT` kali), `preview_summary`, dan `full_text` digabungkan menjadi satu teks per baris (`combined_raw`). Judul diulang karena biasanya lebih padat sinyal dibanding isi artikel sehingga kata-katanya perlu bobot frekuensi lebih besar saat nanti dianalisis berbasis hitungan kata (bag-of-words/TF-IDF).

In [42]:
# Tidak ada nilai kosong yang mengganggu penggabungan teks.
news[["title", "preview_summary", "full_text"]] = (
    news[["title", "preview_summary", "full_text"]].fillna("")
)

# Gabungkan title (diulang TITLE_REPEAT_WEIGHT kali) + preview_summary + full_text.
news["combined_raw"] = (
    ((news["title"] + " ") * TITLE_REPEAT_WEIGHT)
    + news["preview_summary"] + " "
    + news["full_text"]
)

print(f"Jumlah baris: {len(news)}")
news[["title", "combined_raw"]].head(3)

Jumlah baris: 4877


,title,combined_raw
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...
1,Oil rises 2% as White House says no US-Iran talks happening,Oil rises 2% as White House says no US-Iran talks happening Oil rises 2% as White House says no US-Iran talks happen...
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ... Jamie Dimon says...


### 6.2 Pengecekan Simbol pada Data

Dilakukan pengecekan simbol non-alfanumerik apa saja yang benar-benar muncul di `combined_raw` beserta frekuensinya untuk menentukan simbol apa yang diubah menjadi kata, mengingat simbol, seperti mata uang dapat menjadi informasi yang penting pada konteks ini.

In [43]:
import unicodedata
from collections import Counter

symbol_counter = Counter()
for text in news["combined_raw"]:
    for ch in text:
        if not (ch.isalnum() or ch.isspace()):
            symbol_counter[ch] += 1

symbol_df = pd.DataFrame(
    [
        (ch, unicodedata.name(ch, "UNKNOWN"), count)
        for ch, count in symbol_counter.most_common()
    ],
    columns=["simbol", "nama_unicode", "jumlah"],
)

print(f"Jumlah jenis simbol unik pada combined_raw: {len(symbol_df)}")
symbol_df.head(20)

Jumlah jenis simbol unik pada combined_raw: 55


,simbol,nama_unicode,jumlah
0,.,FULL STOP,248193
1,",",COMMA,213004
2,"""",QUOTATION MARK,92029
3,',APOSTROPHE,62730
4,-,HYPHEN-MINUS,44521
5,%,PERCENT SIGN,21061
6,$,DOLLAR SIGN,11034
7,—,EM DASH,10700
8,’,RIGHT SINGLE QUOTATION MARK,7249
9,:,COLON,5278


### 6.3 Normalization & Cleaning

`combined_raw` dibersihkan dari elemen yang tidak membawa informasi tekstual: entitas HTML (`&amp;`, dst.), tag HTML, URL, alamat email, dan emoji. Berdasarkan pengecekan, hanya simbol mata uang (`%`, `$`, `£`, `€`, `¥`) yang diubah jadi kata ("percent", "dollar", "pound", "euro", "yen") supaya konteksnya tetap terbaca. Simbol lain seperti `&` dan `+`, serta tanda baca umum (titik, koma, kutip, tanda pisah, dll.) dibuang seperti simbol non-alfanumerik lainnya.

In [44]:
URL_RE = re.compile(r"http\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
HTML_TAG_RE = re.compile(r"<[^>]+>")
EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "\U00002600-\U000026FF"
    "]+",
    flags=re.UNICODE,
)

# Hanya simbol mata uang yang diubah jadi kata.
# Simbol lain (&, +, tanda baca umum) dibuang di NON_ALNUM_RE.
CURRENCY_WORDS = {
    "%": "percent",
    "$": "dollar",
    "£": "pound",
    "€": "euro",
    "¥": "yen",
}
CURRENCY_RE = re.compile("|".join(re.escape(sym) for sym in CURRENCY_WORDS))
NON_ALNUM_RE = re.compile(r"[^a-zA-Z0-9\s]")  # simbol/tanda baca sisa dibuang; huruf & angka dipertahankan
MULTI_SPACE_RE = re.compile(r"\s+")


def normalize_text(text: str) -> str:
    """Hapus HTML/URL/email/emoji, ubah simbol mata uang jadi kata, angka dipertahankan."""
    if not isinstance(text, str) or not text.strip():
        return ""
    t = html.unescape(text)
    t = HTML_TAG_RE.sub(" ", t)
    t = URL_RE.sub(" ", t)
    t = EMAIL_RE.sub(" ", t)
    t = EMOJI_RE.sub(" ", t)
    t = CURRENCY_RE.sub(lambda m: f" {CURRENCY_WORDS[m.group(0)]} ", t)
    t = NON_ALNUM_RE.sub(" ", t)
    t = MULTI_SPACE_RE.sub(" ", t).strip()
    return t


news["normalized_text"] = news["combined_raw"].apply(normalize_text)

news[["combined_raw", "normalized_text"]].head(3)

,combined_raw,normalized_text
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...
1,Oil rises 2% as White House says no US-Iran talks happening Oil rises 2% as White House says no US-Iran talks happen...,Oil rises 2 percent as White House says no US Iran talks happening Oil rises 2 percent as White House says no US Ira...
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ... Jamie Dimon says...,Jamie Dimon says markets underestimate risks and he wouldn t buy stocks or Treasurys at current Jamie Dimon says mar...


### 6.4 Lowercasing

`normalized_text` diubah menjadi huruf kecil semua, supaya kata yang sama tidak dihitung sebagai token berbeda hanya karena perbedaan kapitalisasi, seperti "Oil" vs "oil".

In [45]:
news["lower_text"] = news["normalized_text"].str.lower()

news[["normalized_text", "lower_text"]].head(3)

,normalized_text,lower_text
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...
1,Oil rises 2 percent as White House says no US Iran talks happening Oil rises 2 percent as White House says no US Ira...,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...
2,Jamie Dimon says markets underestimate risks and he wouldn t buy stocks or Treasurys at current Jamie Dimon says mar...,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...


### 6.5 Language Detection

Dataset diambil dari CNBC (berbahasa Inggris), tetapi beberapa baris berpotensi berisi teks yang rusak, terlalu pendek, atau bercampur bahasa lain akibat proses scraping. Tahap ini mendeteksi bahasa dari `lower_text` menggunakan `langdetect`, lalu mempertahankan hanya baris yang terdeteksi berbahasa Inggris (`en`) agar konsisten dengan resource NLP berikutnya (stopwords & lemmatizer Bahasa Inggris).

In [46]:
def detect_language(text: str) -> str:
    if not isinstance(text, str) or len(text.strip()) < 20:
        return "unknown"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"


news["detected_lang"] = news["lower_text"].apply(detect_language)

print("Distribusi bahasa terdeteksi:")
print(news["detected_lang"].value_counts().head(10))

baris_sebelum = len(news)
news = news.loc[news["detected_lang"] == "en"].reset_index(drop=True)
print(f"\nSetelah filter bahasa Inggris (en) : {len(news)} (dibuang {baris_sebelum - len(news)} baris)")

news[["lower_text", "detected_lang"]].head(3)

Distribusi bahasa terdeteksi:
detected_lang
en    4877
Name: count, dtype: int64

Setelah filter bahasa Inggris (en) : 4877 (dibuang 0 baris)


,lower_text,detected_lang
0,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...,en
1,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...,en
2,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...,en


### 6.6 Stopword Removal

`lower_text` ditokenisasi sederhana, seperti split spasi, lalu kata-kata umum Bahasa Inggris yang tidak membawa makna signifikan (*stopwords* seperti "the", "and", "is") dibuang menggunakan korpus `stopwords` dari NLTK. Ditambahkan pula beberapa *custom stopword*, seperti "cnbc", "article". Token dengan panjang 1 karakter turut dibuang karena dianggap sisa tanda baca/noise.

In [47]:
CUSTOM_STOPWORDS = {
    "cnbc", "article", "read", "said", "says", "say",
    "would", "could", "also", "get", "got",
}
STOPWORDS_EN = set(stopwords.words("english")) | CUSTOM_STOPWORDS


def remove_stopwords(text: str) -> list:
    # Tokenisasi split-spasi lalu buang stopword Bahasa Inggris & token 1 karakter.
    if not isinstance(text, str) or not text.strip():
        return []
    tokens = text.split()
    return [
        tok for tok in tokens
        if tok not in STOPWORDS_EN and (len(tok) > 1 or tok.isdigit())
    ]


news["tokens_no_stopwords"] = news["lower_text"].apply(remove_stopwords)
news["text_no_stopwords"] = news["tokens_no_stopwords"].apply(" ".join)

avg_before = news["lower_text"].apply(word_count).mean()
avg_after = news["tokens_no_stopwords"].apply(len).mean()
print(f"Rata-rata jumlah kata sebelum stopword removal : {avg_before:.1f}")
print(f"Rata-rata jumlah kata sesudah stopword removal  : {avg_after:.1f}")

news[["lower_text", "text_no_stopwords"]].head(3)

Rata-rata jumlah kata sebelum stopword removal : 807.1
Rata-rata jumlah kata sesudah stopword removal  : 473.3


,lower_text,text_no_stopwords
0,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...,central banks bringing gold reserves home asgeopoliticalrisks rise central banks bringing gold reserves home asgeopo...
1,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...,oil rises 2 percent white house us iran talks happening oil rises 2 percent white house us iran talks happening oil ...
2,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...,jamie dimon markets underestimate risks buy stocks treasurys current jamie dimon markets underestimate risks buy sto...


### 6.7 Lemmatization (POS-aware)

`tokens_no_stopwords` dilemmatisasi menggunakan `WordNetLemmatizer` untuk mereduksi tiap token ke bentuk dasarnya (*lemma*), seperti "banks" → "bank", "bringing" → "bring". `WordNetLemmatizer.lemmatize(token)` tanpa parameter `pos` akan otomatis mengasumsikan token adalah kata benda (default `pos="n"`). Akibatnya kata kerja seperti "rising" atau "rose" justru tidak tereduksi ke bentuk dasarnya ("rise") sehingga lemmatizer menganggapnya sudah bentuk dasar. Supaya hasilnya benar, setiap token terlebih dulu diberi POS tag lewat `nltk.pos_tag`, dipetakan ke format POS WordNet (*noun, verb, adjective, adverb*), baru dilemmatisasi dengan `pos` yang sesuai.

In [48]:
lemmatizer = WordNetLemmatizer()

POS_MAP = {
    "J": wordnet.ADJ,
    "V": wordnet.VERB,
    "N": wordnet.NOUN,
    "R": wordnet.ADV,
}


def get_wordnet_pos(treebank_tag: str) -> str:
    # Tag POS Penn Treebank (huruf pertama) ke format POS WordNet; default kata benda.
    return POS_MAP.get(treebank_tag[0], wordnet.NOUN)


def lemmatize_tokens(tokens: list) -> list:
    # Lemmatisasi tiap token berdasarkan POS tag-nya (POS-aware lemmatization).
    if not tokens:
        return []
    tagged_tokens = pos_tag(tokens)
    return [
        lemmatizer.lemmatize(token, pos=get_wordnet_pos(tag))
        for token, tag in tagged_tokens
    ]


news["tokens_lemmatized"] = news["tokens_no_stopwords"].apply(lemmatize_tokens)
news["text_lemmatized"] = news["tokens_lemmatized"].apply(" ".join)

news[["text_no_stopwords", "text_lemmatized"]].head(3)

,text_no_stopwords,text_lemmatized
0,central banks bringing gold reserves home asgeopoliticalrisks rise central banks bringing gold reserves home asgeopo...,central bank bring gold reserve home asgeopoliticalrisks rise central bank bring gold reserve home asgeopoliticalris...
1,oil rises 2 percent white house us iran talks happening oil rises 2 percent white house us iran talks happening oil ...,oil rise 2 percent white house u iran talk happen oil rise 2 percent white house u iran talk happen oil price rise t...
2,jamie dimon markets underestimate risks buy stocks treasurys current jamie dimon markets underestimate risks buy sto...,jamie dimon market underestimate risk buy stock treasurys current jamie dimon market underestimate risk buy stock tr...


## 7. Standardisasi Zona Waktu (WIB) & Aturan Market Cut-off (T+1)

Tahapan:
1. **Standardisasi zona waktu**: `date_raw_dt` diasumsikan berada di zona waktu sumber (`SOURCE_TZ = America/New_York`, zona waktu acuan CNBC/NYSE). Jika timestamp masih *naive* (belum ada offset), timestamp di-*localize* terlebih dahulu ke `SOURCE_TZ`, baru dikonversi ke `TARGET_TZ = Asia/Jakarta` (WIB). Jika timestamp sudah *tz-aware* (memiliki offset, misal dari `published_iso`), konversi dilakukan langsung tanpa localize ulang. Langkah ini menggantikan konversi sementara pada bagian 3.2 yang masih mengasumsikan input sebagai UTC.
2. **Aturan market cut-off (T+1)**: Kurs referensi Bank Indonesia sudah final berdasarkan transaksi pasar sebelum jam `MARKET_CUTOFF_HOUR` (15:00 WIB). Berita yang terbit **setelah** jam cut-off tersebut tidak mungkin secara kausal memengaruhi kurs pada hari kalender yang sama karena nilainya sudah ditetapkan lebih dulu, sehingga berita tersebut diberi label tanggal target di hari kalender berikutnya (T+1). Berita yang terbit sebelum cut-off tetap berada pada tanggal target yang sama (T+0).


In [49]:
# --- 7.1 Standardisasi zona waktu berdasarkan SOURCE_TZ -> TARGET_TZ ---
def to_target_tz(dt_series: pd.Series, source_tz: str, target_tz: str) -> pd.Series:
    """Konversi Series datetime ke target_tz.
    Jika naive (belum ada offset), asumsikan berada di source_tz lalu localize.
    Jika sudah tz-aware, langsung convert ke target_tz.
    """
    dt_series = pd.to_datetime(dt_series, errors="coerce")
    is_naive = dt_series.dt.tz is None

    if is_naive:
        localized = dt_series.dt.tz_localize(
            source_tz, ambiguous="NaT", nonexistent="NaT"
        )
    else:
        localized = dt_series

    return localized.dt.tz_convert(target_tz).dt.tz_localize(None)


news["date_raw_dt_wib"] = to_target_tz(news["date_raw_dt"], SOURCE_TZ, TARGET_TZ)
news["tanggal"] = news["date_raw_dt_wib"].dt.strftime("%m/%d/%Y")
news["jam"] = news["date_raw_dt_wib"].dt.strftime("%I:%M %p")

jumlah_gagal_tz = news["date_raw_dt_wib"].isna().sum()
print(f"Baris gagal konversi zona waktu     : {jumlah_gagal_tz}")

# --- 7.2 Aturan market cut-off (T+1) ---
news["jam_wib"] = news["date_raw_dt_wib"].dt.hour + news["date_raw_dt_wib"].dt.minute / 60
news["after_cutoff"] = news["jam_wib"] >= MARKET_CUTOFF_HOUR

news["target_date_raw"] = news["date_raw_dt_wib"].dt.normalize()
news.loc[news["after_cutoff"], "target_date_raw"] += pd.Timedelta(days=1)

print(
    f"Berita setelah jam cut-off ({MARKET_CUTOFF_HOUR:.0f}:00 WIB) -> digeser ke T+1: "
    f"{news['after_cutoff'].sum()} dari {len(news)} baris"
)

news[["title", "date_raw_dt_wib", "after_cutoff", "target_date_raw"]].head(5)


Baris gagal konversi zona waktu     : 0
Berita setelah jam cut-off (15:00 WIB) -> digeser ke T+1: 1478 dari 4877 baris


,title,date_raw_dt_wib,after_cutoff,target_date_raw
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise,2026-06-18 01:28:36,False,2026-06-18
1,Oil rises 2% as White House says no US-Iran talks happening,2026-08-27 20:49:03,True,2026-08-28
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,2026-07-21 17:01:01,True,2026-07-22
3,Treasury yields edge higher as investors mapgeopoliticalrisks,2026-07-22 02:25:56,False,2026-07-22
4,China’s super-rich fled Singapore. Now they want to come back,2026-08-28 17:29:56,True,2026-08-29


## 8. Penyelarasan Hari Libur/Akhir Pekan (Forward-Rolling Alignment)

Tahapan:
1. **Daftar non-trading days**: gunakan library `holidays` untuk mendapatkan hari libur nasional Indonesia (`holidays.Indonesia`) pada rentang `START_DATE`–`END_DATE`, digabung dengan hari Sabtu/Minggu (akhir pekan), serta tanggal yang secara faktual tidak tersedia pada data kurs BI (`bi["Tanggal"]`). Gabungan ketiganya menghasilkan himpunan `NON_TRADING_DAYS`.
2. **Forward-rolling alignment**: apabila `target_date_raw` hasil tahap 7 jatuh pada non-trading day, tanggal digeser maju (*forward-roll*) hari demi hari sampai menemukan hari bursa (*trading day*) terdekat berikutnya. Arah pergeseran selalu **maju**, tidak pernah mundur, karena secara kausal berita tidak mungkin memengaruhi kurs yang sudah terjadi sebelum berita itu terbit.
3. Hasil akhir disimpan pada kolom `target_date`, yaitu tanggal kurs BI yang menjadi *join key* berita tersebut pada tahap agregasi & merge berikutnya (bagian 9–10).


In [53]:
# KONFIGURASI
CUTOFF_HOUR = 15

# 7.x — Pastikan tipe data datetime
news["date_raw_dt_wib"] = pd.to_datetime(news["date_raw_dt_wib"])
bi["Tanggal"] = pd.to_datetime(bi["Tanggal"])

# 7.y — Tentukan after_cutoff & target_date_raw
# after_cutoff = True jika berita terbit pada/lebih dari jam cutoff
news["after_cutoff"] = news["date_raw_dt_wib"].dt.hour >= CUTOFF_HOUR

# target_date_raw: tanggal dasar (sebelum forward-roll ke trading day)
news["target_date_raw"] = news["date_raw_dt_wib"].dt.normalize()

# Jika after_cutoff True -> geser ke hari berikutnya
news.loc[news["after_cutoff"], "target_date_raw"] += pd.Timedelta(days=1)

print("Contoh hasil after_cutoff & target_date_raw:")
print(news[["date_raw_dt_wib", "after_cutoff", "target_date_raw"]].head(10))
print()

# 8.1 — Bangun himpunan non-trading day
years = range(START_DATE.year, END_DATE.year + 1)
id_holidays = holidays.Indonesia(years=years)

trading_dates_bi = set(bi["Tanggal"].dt.normalize())
all_dates = pd.date_range(START_DATE.normalize(), END_DATE.normalize(), freq="D")

NON_TRADING_DAYS = {
    d for d in all_dates
    if d.weekday() >= 5            # Sabtu(5)/Minggu(6)
    or d in id_holidays            # libur nasional
    or d not in trading_dates_bi   # tidak ada data kurs BI pada tanggal tsb
}

print(f"Total hari pada rentang     : {len(all_dates)}")
print(f"Jumlah non-trading days     : {len(NON_TRADING_DAYS)}")
print(f"Jumlah trading days         : {len(all_dates) - len(NON_TRADING_DAYS)}")
print()

# 8.2 — Forward-rolling alignment
def forward_roll(date: pd.Timestamp, non_trading: set, max_shift: int = 10) -> pd.Timestamp:
    """Geser tanggal maju sampai menemukan trading day terdekat berikutnya."""
    shifted = date
    for _ in range(max_shift):
        if shifted not in non_trading:
            return shifted
        shifted += pd.Timedelta(days=1)
    return pd.NaT  # gagal menemukan trading day dalam batas max_shift


news["target_date"] = news["target_date_raw"].apply(
    lambda d: forward_roll(d, NON_TRADING_DAYS) if pd.notna(d) else pd.NaT
)

jumlah_gagal_roll = news["target_date"].isna().sum()
jumlah_bergeser = (news["target_date"] != news["target_date_raw"]).sum()

print(f"Baris gagal forward-roll (melebihi batas): {jumlah_gagal_roll}")
print(f"Baris yang tergeser ke trading day berikutnya: {jumlah_bergeser}")

news[["title", "target_date_raw", "target_date", "after_cutoff"]].head(5)

Contoh hasil after_cutoff & target_date_raw:
      date_raw_dt_wib  after_cutoff target_date_raw
0 2026-06-18 01:28:36         False      2026-06-18
1 2026-08-27 20:49:03          True      2026-08-28
2 2026-07-21 17:01:01          True      2026-07-22
3 2026-07-22 02:25:56         False      2026-07-22
4 2026-08-28 17:29:56          True      2026-08-29
5 2026-08-18 06:19:50         False      2026-08-18
6 2026-05-28 05:39:45         False      2026-05-28
7 2026-06-16 00:41:45         False      2026-06-16
8 2026-05-22 00:35:37         False      2026-05-22
9 2026-08-20 03:32:56         False      2026-08-20

Total hari pada rentang     : 1827
Jumlah non-trading days     : 625
Jumlah trading days         : 1202

Baris gagal forward-roll (melebihi batas): 22
Baris yang tergeser ke trading day berikutnya: 1435


,title,target_date_raw,target_date,after_cutoff
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise,2026-06-18,2026-06-18,False
1,Oil rises 2% as White House says no US-Iran talks happening,2026-08-28,2026-08-28,True
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,2026-07-22,2026-07-22,True
3,Treasury yields edge higher as investors mapgeopoliticalrisks,2026-07-22,2026-07-22,False
4,China’s super-rich fled Singapore. Now they want to come back,2026-08-29,2026-08-31,True
